In [ ]:
!pip install requests beautifulsoup4 langdetect

In [ ]:
# ============================================================
# LinkedIn Job Scraper - No AI API Required
# Google Colab / Python 3 Compatible
# ============================================================
# Run this first:
#   !pip install requests beautifulsoup4 langdetect
# ============================================================


import requests
from bs4 import BeautifulSoup
from langdetect import detect, LangDetectException
import json
import time
from dataclasses import dataclass, asdict


# ============================================================
# >>>  CONFIGURATION - EDIT THIS SECTION FOR YOUR FIELD  <<<
# ============================================================
#
# Replace the list below with job titles relevant to YOUR field.
#
# Examples for other fields:
#
#   SOFTWARE ENGINEERING:
#     JOB_KEYWORDS = [
#         "Software Engineer", "Backend Developer", "Frontend Developer",
#         "Full Stack Developer", "Python Developer", "Java Developer",
#         "DevOps Engineer", "Cloud Engineer", "Site Reliability Engineer",
#     ]
#
#   MECHANICAL ENGINEERING:
#     JOB_KEYWORDS = [
#         "Mechanical Engineer", "Design Engineer", "Product Engineer",
#         "Manufacturing Engineer", "Structural Engineer", "CAD Engineer",
#         "Thermodynamics Engineer", "HVAC Engineer", "Automotive Engineer",
#     ]
#
#   DATA / AI:
#     JOB_KEYWORDS = [
#         "Data Engineer", "Data Scientist", "Machine Learning Engineer",
#         "AI Engineer", "Data Analyst", "Business Intelligence Engineer",
#         "MLOps Engineer", "NLP Engineer",
#     ]
#
# ---- ELECTRICAL ENGINEERING (default) ----------------------

JOB_KEYWORDS = [
    # Core EE titles
    "Electrical Engineer",
    "Electronics Engineer",
    "Power Engineer",
    "Embedded Systems Engineer",
    "Control Systems Engineer",
    "Instrumentation Engineer",
    "Automation Engineer",

    # Power & energy
    "Power Systems Engineer",
    "High Voltage Engineer",
    "Substation Engineer",
    "Protection and Relay Engineer",
    "Renewable Energy Engineer",

    # Electronics & hardware
    "PCB Design Engineer",
    "Hardware Engineer",
    "Circuit Design Engineer",
    "Analog Design Engineer",
    "RF Engineer",
    "Signal Integrity Engineer",

    # Embedded & firmware
    "Firmware Engineer",
    "Embedded Software Engineer",
    "FPGA Engineer",
    "PLC Programmer",

    # Related roles
    "Electrical Design Engineer",
    "Electrical Project Engineer",
    "Electrical Site Engineer",
    "Commissioning Engineer",
    "Field Service Engineer",
    "Test and Measurement Engineer",
]

# ---- DELAY SETTINGS (no need to change these normally) -----

DELAY_SEARCH = 2.5   # Seconds between page requests
DELAY_DESC   = 1.5   # Seconds between individual job description fetches

# ============================================================
# END OF CONFIGURATION
# ============================================================


# ============================================================
# FILTER REFERENCE TABLES
# ============================================================

EXPERIENCE_LEVELS = {
    "1": ("Internship",       "1"),
    "2": ("Entry level",      "2"),
    "3": ("Associate",        "3"),
    "4": ("Mid-Senior level", "4"),
    "5": ("Director",         "5"),
    "6": ("Executive",        "6"),
}

DATE_POSTED = {
    "1": ("Any time",      ""),
    "2": ("Past month",    "r2592000"),
    "3": ("Past week",     "r604800"),
    "4": ("Past 24 hours", "r86400"),
}

SUPPORTED_LANGUAGES = {
    "1":  ("English",    "en"),
    "2":  ("Dutch",      "nl"),
    "3":  ("German",     "de"),
    "4":  ("French",     "fr"),
    "5":  ("Spanish",    "es"),
    "6":  ("Italian",    "it"),
    "7":  ("Portuguese", "pt"),
    "8":  ("Polish",     "pl"),
    "9":  ("Swedish",    "sv"),
    "10": ("Danish",     "da"),
    "11": ("Norwegian",  "no"),
    "12": ("Finnish",    "fi"),
    "13": ("Arabic",     "ar"),
    "14": ("Turkish",    "tr"),
    "15": ("Romanian",   "ro"),
}


# ============================================================
# DATA CLASS
# ============================================================

@dataclass
class Job:
    title:                str
    company:              str
    location:             str
    posted:               str
    job_url:              str
    job_id:               str
    language:             str = "unknown"
    description_snippet:  str = ""


# ============================================================
# SCRAPER FUNCTIONS
# ============================================================

def build_search_url(
    keyword: str,
    location: str,
    experience_code: str,
    date_code: str,
    start: int = 0,
) -> str:
    """Build a LinkedIn guest job search URL with filters."""
    base = "https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search"
    params = {"keywords": keyword, "location": location, "start": start}
    if experience_code:
        params["f_E"] = experience_code
    if date_code:
        params["f_TPR"] = date_code
    param_str = "&".join(
        f"{k}={requests.utils.quote(str(v))}" for k, v in params.items()
    )
    return f"{base}?{param_str}"


def fetch_job_listings(url: str, headers: dict) -> list:
    """Fetch a page of job cards from LinkedIn search results."""
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  [!] Failed to fetch listings: {e}")
        return []

    soup  = BeautifulSoup(resp.text, "html.parser")
    cards = soup.find_all("div", class_="base-card")
    jobs  = []

    for card in cards:
        try:
            title_tag    = card.find("h3", class_="base-search-card__title")
            company_tag  = card.find("h4", class_="base-search-card__subtitle")
            location_tag = card.find("span", class_="job-search-card__location")
            posted_tag   = card.find("time")
            link_tag     = card.find("a", class_="base-card__full-link")
            job_id       = card.get("data-entity-urn", "").split(":")[-1]

            jobs.append(Job(
                title    = title_tag.get_text(strip=True)    if title_tag    else "N/A",
                company  = company_tag.get_text(strip=True)  if company_tag  else "N/A",
                location = location_tag.get_text(strip=True) if location_tag else "N/A",
                posted   = posted_tag.get_text(strip=True)   if posted_tag   else "N/A",
                job_url  = link_tag["href"].strip()           if link_tag     else "N/A",
                job_id   = job_id,
            ))
        except Exception as e:
            print(f"  [!] Skipped a card: {e}")

    return jobs


def fetch_job_description(job_url: str, headers: dict) -> str:
    """Fetch full job description text from an individual job page."""
    try:
        resp = requests.get(job_url, headers=headers, timeout=12)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        desc_div = soup.find("div", class_="description__text")
        if desc_div:
            return desc_div.get_text(separator=" ", strip=True)

        # Fallback: grab all paragraphs and list items
        paragraphs = soup.find_all(["p", "li"])
        return " ".join(p.get_text(strip=True) for p in paragraphs)

    except requests.RequestException as e:
        print(f"    [!] Could not fetch job page: {e}")
        return ""


def detect_language(text: str) -> str:
    """Detect the language of a text string. Returns an ISO code or 'unknown'."""
    if not text or len(text.strip()) < 30:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


# ============================================================
# MAIN SCRAPER
# ============================================================

def scrape_jobs(
    keywords:          list,
    location:          str,
    experience_code:   str,
    date_code:         str,
    target_lang_code:  str,
    max_pages:         int   = 2,
    request_delay:     float = 2.5,
    desc_delay:        float = 1.5,
    save_to_json:      bool  = True,
) -> list:
    """
    Iterates over all keywords, scrapes LinkedIn job listings,
    fetches each job description, detects its language, and
    keeps only jobs written in the target language.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    }

    lang_label = SUPPORTED_LANGUAGES.get(target_lang_code, target_lang_code)
    all_jobs   = []
    seen_ids   = set()

    print("\n" + "=" * 60)
    print("SCRAPING STARTED")
    print("=" * 60)

    for keyword in keywords:
        print(f"\n[SEARCH] '{keyword}' in {location}")

        for page in range(max_pages):
            start = page * 25
            url   = build_search_url(keyword, location, experience_code, date_code, start)
            print(f"  Page {page + 1} ... ", end="", flush=True)

            listings = fetch_job_listings(url, headers)
            if not listings:
                print("no results or end of pages.")
                break
            print(f"{len(listings)} listings found.")

            for job in listings:
                if not job.job_id or job.job_id in seen_ids:
                    continue
                seen_ids.add(job.job_id)

                print(f"    Checking: {job.title[:50]:<50} ", end="", flush=True)
                desc         = fetch_job_description(job.job_url, headers)
                lang         = detect_language(desc)
                job.language = lang
                job.description_snippet = desc[:300]

                if lang == target_lang_code:
                    all_jobs.append(job)
                    print(f"[SAVED - {lang_label}]")
                else:
                    print(f"[SKIPPED - {lang}]")

                time.sleep(desc_delay)

            time.sleep(request_delay)

    print(f"\n[DONE] Total {lang_label} jobs found: {len(all_jobs)}")

    if save_to_json and all_jobs:
        safe_location = location.replace(" ", "_").replace(",", "")
        filename = f"linkedin_jobs_{safe_location}_{target_lang_code}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump([asdict(j) for j in all_jobs], f, indent=2, ensure_ascii=False)
        print(f"[SAVED] Results saved to: {filename}")

    return all_jobs


# ============================================================
# PRINT RESULTS
# ============================================================

def print_results(jobs: list, lang_label: str) -> None:
    """Print all found jobs in a clean readable format."""
    print("\n" + "=" * 72)
    print(f"RESULTS - {lang_label.upper()} JOB POSTINGS".center(72))
    print("=" * 72)

    if not jobs:
        print("\nNo jobs found matching your criteria.")
        print("Suggestions:")
        print("  - Try 'Any time' for the date filter")
        print("  - Search a whole country instead of a specific city")
        print("  - Try a different experience level")
        return

    for i, job in enumerate(jobs, 1):
        print(f"\n[{i}] {job.title}")
        print(f"    Company  : {job.company}")
        print(f"    Location : {job.location}")
        print(f"    Posted   : {job.posted}")
        print(f"    URL      : {job.job_url}")
        if job.description_snippet:
            snippet = job.description_snippet[:200].replace("\n", " ")
            print(f"    Preview  : {snippet}...")

    print("\n" + "=" * 72)
    print(f"Total: {len(jobs)} {lang_label} job(s) found.")
    print("=" * 72)


# ============================================================
# INTERACTIVE SETUP PROMPTS
# ============================================================

def ask_experience_level() -> tuple:
    """Ask user to select experience level. Returns (label, code)."""
    print("=" * 60)
    print("STEP 1 - Select your experience level:")
    print("=" * 60)
    for key, (label, _) in EXPERIENCE_LEVELS.items():
        print(f"  {key}. {label}")
    print()
    while True:
        choice = input("  Enter your choice (1-6): ").strip()
        if choice in EXPERIENCE_LEVELS:
            label, code = EXPERIENCE_LEVELS[choice]
            print(f"  >> Selected: {label}\n")
            return label, code
        print("  Invalid choice. Please enter a number between 1 and 6.")


def ask_language() -> tuple:
    """Ask user which language to filter job postings by. Returns (label, code)."""
    print("=" * 60)
    print("STEP 2 - Select preferred job posting language:")
    print("=" * 60)
    for key, (label, _) in SUPPORTED_LANGUAGES.items():
        print(f"  {key:>2}. {label}")
    print()
    while True:
        choice = input("  Enter your choice (1-15): ").strip()
        if choice in SUPPORTED_LANGUAGES:
            label, code = SUPPORTED_LANGUAGES[choice]
            print(f"  >> Selected: {label} ({code})\n")
            return label, code
        print("  Invalid choice. Please enter a number between 1 and 15.")


def ask_date_posted() -> tuple:
    """Ask user to select date posted filter. Returns (label, code)."""
    print("=" * 60)
    print("STEP 3 - Select date posted:")
    print("=" * 60)
    for key, (label, _) in DATE_POSTED.items():
        print(f"  {key}. {label}")
    print()
    while True:
        choice = input("  Enter your choice (1-4): ").strip()
        if choice in DATE_POSTED:
            label, code = DATE_POSTED[choice]
            print(f"  >> Selected: {label}\n")
            return label, code
        print("  Invalid choice. Please enter a number between 1 and 4.")


def ask_location() -> str:
    """Ask user for country and optional city. Returns formatted location string."""
    print("=" * 60)
    print("STEP 4 - Enter location:")
    print("=" * 60)
    while True:
        country = input("  Enter country (e.g. Germany, Italy, Netherlands): ").strip()
        if country:
            break
        print("  Country cannot be empty. Please try again.")
    city = input("  Enter city (press Enter to search the whole country): ").strip()
    location = f"{city}, {country}" if city else country
    print(f"  >> Searching in: {location}\n")
    return location


def ask_max_pages() -> int:
    """Ask user how many pages to scrape per keyword. No upper limit."""
    print("=" * 60)
    print("STEP 5 - How many pages per keyword?")
    print("=" * 60)
    print("  Each page contains up to 25 job listings.")
    print("  Recommended: 1-2 pages for speed, 3-5 for more results.")
    print("  If you enter more pages than LinkedIn has, it stops automatically.")
    print()
    while True:
        choice = input("  Enter number of pages (minimum 1): ").strip()
        if choice.isdigit() and int(choice) >= 1:
            pages = int(choice)
            print(f"  >> Fetching up to {pages} page(s) per keyword\n")
            return pages
        print("  Invalid choice. Please enter a whole number greater than 0.")


# ============================================================
# ENTRY POINT
# ============================================================

def main():
    print("\n" + "=" * 60)
    print("  LinkedIn Job Scraper")
    print("=" * 60)
    print(f"  Searching across {len(JOB_KEYWORDS)} job title keywords.")
    print("=" * 60 + "\n")

    exp_label,  exp_code  = ask_experience_level()
    lang_label, lang_code = ask_language()
    date_label, date_code = ask_date_posted()
    location              = ask_location()
    max_pages             = ask_max_pages()

    print("=" * 60)
    print("SUMMARY - Starting search with these settings:")
    print("=" * 60)
    print(f"  Experience level : {exp_label}")
    print(f"  Language         : {lang_label} ({lang_code})")
    print(f"  Date posted      : {date_label}")
    print(f"  Location         : {location}")
    print(f"  Pages/keyword    : {max_pages}")
    print(f"  Keywords         : {len(JOB_KEYWORDS)} search terms")
    print(f"  Max results      : ~{max_pages * 25 * len(JOB_KEYWORDS)} (before dedup + filter)")
    print("=" * 60)
    input("\nPress Enter to start scraping...")

    jobs = scrape_jobs(
        keywords         = JOB_KEYWORDS,
        location         = location,
        experience_code  = exp_code,
        date_code        = date_code,
        target_lang_code = lang_code,
        max_pages        = max_pages,
        request_delay    = DELAY_SEARCH,
        desc_delay       = DELAY_DESC,
        save_to_json     = True,
    )

    print_results(jobs, lang_label)


main()